# Research 3 — GPU boundary feasibility only

Select A100/high RAM only within the authorized 50 compute units. Read current rate/balance immediately after allocation. These trusted model-free probes test actual hardware and cgroup facilities. No Drive mount, model download, adapter training or agentic evaluation. Disconnect/delete this newly created test runtime after capturing results if required confinement is unavailable.


In [ ]:
"""Trusted, model-free Colab inventory. This is not a worker isolation gate."""
import json
import os
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def command(args):
    try:
        result = subprocess.run(args, capture_output=True, text=True, timeout=10)
        return {"exit_code": result.returncode, "stdout": result.stdout[:8192],
                "stderr": result.stderr[:2048]}
    except (OSError, subprocess.TimeoutExpired) as error:
        return {"error": type(error).__name__}


def inventory():
    memory = {}
    if Path('/proc/meminfo').exists():
        for line in Path('/proc/meminfo').read_text().splitlines():
            key, value = line.split(':', 1)
            if key in ('MemTotal', 'MemAvailable', 'SwapTotal'):
                memory[key + '_kib'] = int(value.strip().split()[0])
    root = '/content' if Path('/content').exists() else '.'
    gpu = command(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                   '--format=csv,noheader'])
    # Probe namespace availability only. No network or host configuration changes.
    namespace = command(['unshare', '--user', '--map-root-user', '--net',
                         '--pid', '--fork', '/usr/bin/true'])
    mounts = Path('/proc/self/mountinfo').read_text() if Path('/proc/self/mountinfo').exists() else ''
    return {
        'utc': datetime.now(timezone.utc).isoformat(),
        'python': platform.python_version(), 'platform': platform.platform(),
        'uid': os.getuid() if hasattr(os, 'getuid') else None,
        'cpu_count': os.cpu_count(), 'memory': memory,
        'disk_free_bytes': shutil.disk_usage(root).free,
        'gpu': gpu, 'namespace_availability_probe': namespace,
        'bubblewrap_path': shutil.which('bwrap'),
        'drive_mount_lines': [line for line in mounts.splitlines()
                              if '/content/drive' in line or 'drivefs' in line],
        # Report names only, never secret values. Parent may be authenticated;
        # worker must have a separate clean environment and restricted filesystem.
        'sensitive_environment_variable_names': sorted(
            name for name in os.environ
            if any(part in name.upper() for part in ('TOKEN', 'SECRET', 'PASSWORD', 'CREDENTIAL'))),
        'isolation_verified': False,
        'model_worker_started': False,
        'notes': ['Namespace availability does not establish credential, filesystem, '
                  'network, CUDA, seccomp, cgroup, or watchdog containment. '
                  'Agentic evaluation remains prohibited until independent worker probes pass.'],
    }



print(json.dumps(inventory(), indent=2))


In [ ]:
"""Trusted model-free check of the facilities needed to port the local boundary."""
import ctypes.util
import json
import os
import shutil
import subprocess
import uuid
from datetime import datetime, timezone
from pathlib import Path


def run(args):
    try:
        result = subprocess.run(args, capture_output=True, text=True, timeout=15)
        return {'exit_code': result.returncode, 'stdout': result.stdout[:8192],
                'stderr': result.stderr[:2048]}
    except (OSError, subprocess.TimeoutExpired) as error:
        return {'error': type(error).__name__}


def main():
    root = Path('/sys/fs/cgroup')
    status = dict(line.split(':', 1) for line in Path('/proc/self/status').read_text().splitlines()
                  if ':' in line)
    cgroup = {'v2': (root / 'cgroup.controllers').exists()}
    for name in ('cgroup.controllers', 'cgroup.subtree_control', 'memory.max',
                 'memory.swap.max', 'cpu.max', 'pids.max'):
        path = root / name
        cgroup[name] = path.read_text().strip() if path.exists() else None
    temporary = root / ('sp-lense-readiness-' + uuid.uuid4().hex)
    try:
        temporary.mkdir()
        cgroup['child_creation'] = 'succeeded'
        cgroup['child_controllers'] = sorted(path.name for path in temporary.iterdir()
                                           if path.name in ('memory.max', 'cpu.max', 'pids.max'))
    except OSError as error:
        cgroup['child_creation'] = {'errno': error.errno, 'error': error.strerror}
    finally:
        if temporary.exists():
            temporary.rmdir()
    # Try mounting a *new*, private cgroup namespace. Failure is evidence about
    # kernel delegation; do not change the host cgroup hierarchy or limits.
    private_cgroup = run(['unshare', '--user', '--map-root-user', '--mount',
                         '--net', '--pid', '--fork', '/bin/sh', '-c',
                         'mount --make-rprivate / && mkdir -p /tmp/sp-lense-cgroup-probe '
                         '&& mount -t cgroup2 none /tmp/sp-lense-cgroup-probe'])
    result = {
        'utc': datetime.now(timezone.utc).isoformat(),
        'pid1_command': Path('/proc/1/comm').read_text().strip(),
        'capabilities': {name: status[name].strip() for name in ('CapEff', 'CapBnd', 'NoNewPrivs', 'Seccomp')},
        'cgroups': cgroup, 'private_cgroup_mount_probe': private_cgroup,
        'seccomp_library': ctypes.util.find_library('seccomp'),
        'bubblewrap': shutil.which('bwrap'),
        'package_candidates': run(['apt-cache', 'policy', 'bubblewrap', 'libseccomp2']),
        'parent_mounts_relevant_to_containment': [line for line in Path('/proc/self/mountinfo').read_text().splitlines()
                                                if any(part in line for part in ('/sys/fs/cgroup', '/content/drive', 'drivefs'))],
        'isolation_verified': False, 'model_worker_started': False,
    }
    print(json.dumps(result, indent=2))


if __name__ == '__main__':
    main()
